# 10 Essential Data Cleaning Techniques
This notebook follows the [KDnuggets guide](https://www.kdnuggets.com/10-essential-data-cleaning-techniques-explained-in-12-minutes) to mastering data wrangling.

### Prerequisites
Make sure you have `pandas`, `numpy`, `scipy`, and `sklearn` installed.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import re

# Load your dataset
df = pd.read_csv('data/messy_data.csv') 
# Note: Using placeholder logic below for demonstration

## 1. Handling Missing Values
We identify missing data, then decide whether to delete or impute.

In [ ]:
# Identification
missing_data = pd.concat([df.isnull().sum(), (df.isnull().sum() / len(df)) * 100], 
                         axis=1, keys=['Count', 'Percentage'])
print(missing_data[missing_data['Count'] > 0])

# Imputation (Median for numeric, Mode for categorical)
df['age'] = df['age'].fillna(df['age'].median())
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])

# Advanced: KNN Imputation
imputer = KNNImputer(n_neighbors=3)
numeric_cols = ['age', 'income', 'customer_rating', 'purchase_amount']
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

## 2. Removing Duplicates
We look for exact row matches and "functional" duplicates (same name/email).

In [ ]:
# Smart deduplication: keep the record with the most data (completeness)
df['completeness'] = df.notna().sum(axis=1)
df = df.sort_values('completeness', ascending=False).drop_duplicates(subset=['name', 'email'])
df = df.drop(columns=['completeness'])

## 3. Standardizing Text Data
Cleaning up inconsistent casing and mapping varied entries (like "US" vs "USA") to a single standard.

In [ ]:
df['name'] = df['name'].str.title()
df['country'] = df['country'].replace({'US': 'USA', 'U.S.A.': 'USA', 'United States': 'USA'})

# Custom function for Education
def standardize_education(edu_str):
    if pd.isna(edu_str): return np.nan
    edu_str = str(edu_str).lower()
    if 'bachelor' in edu_str: return "Bachelor's Degree"
    if 'master' in edu_str: return "Master's Degree"
    return "Other"

df['education'] = df['education'].apply(standardize_education)

## 4. Managing Outliers
Detecting extreme values using Z-score or IQR and applying **Winsorization** (capping).

In [ ]:
# IQR Method
Q1, Q3 = df['income'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

# Winsorization (Capping at 1st and 99th percentiles)
df['income_capped'] = df['income'].clip(lower=df['income'].quantile(0.01), 
                                        upper=df['income'].quantile(0.99))

## 5. & 9. Types and Engineering
Ensuring dates are usable and creating quality indicators.

In [ ]:
# Date Conversion
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')

# Quality Score (0-10)
df['quality_score'] = df.notna().sum(axis=1) / len(df.columns) * 10

# Missing value indicator
df['email_is_missing'] = df['email'].isnull().astype(int)

## Final Clean Dataset
The data is now standardized, imputed, and scaled, making it ready for Machine Learning or Analysis.